# Quick test: visualize volumes, masks, and a tiny reconstruction demo

This notebook is meant to be a **minimal, executable** sanity-check for:

1. Converting a `.tif` to a downsampled `.npy` volume
2. Visualizing the `.npy` volume slices (via `draw_nifti_slices_with_quiver`)
3. Creating a masked pretext sample and visualizing input/target/mask
4. Running a tiny `FishModule` forward pass to see reconstruction inside the mask

In [ ]:
# --- Parameters (edit these first) ---
from pathlib import Path

# Input/Output paths

tif_path = Path("data/raw/3dpf_0601_FLUO4_1_ShiftCorrectionAuto.tif")
# tif_path = Path("data/raw/3dpf_0601_FLUO4_2_ShiftCorrectionAuto_Lateral_Animation.tif")

# Downsample scale (D, H, W)
# scale = (1.0, 1.0, 1.0)
scale = (0.25, 0.25, 0.25)

npy_path = Path(f"data/volume/id_4_1_{scale[0]}.npy")

# Mask parameters
mask_type = "block"  # 'block' or 'patch'
mask_ratio = 0.3


In [ ]:
# --- Imports ---
import numpy as np
import torch

from utils.tif2volume import process_tif_to_volume
from utils.gen_pretext_dataset import create_masked_sample
from utils.visualize_3d import draw_nifti_slices_with_quiver, draw_nifti_slices_with_gt

from utils.load_config import load_config
from model.lightning.fish_module import FishModule

%matplotlib widget
%load_ext autoreload
%autoreload 2

## 1) Convert TIF → NPY (using `utils/tif2volume.py`)

In [ ]:
# Convert the tif to npy if needed (safe to re-run)
assert tif_path.exists(), f"Missing tif: {tif_path}"
npy_path.parent.mkdir(parents=True, exist_ok=True)

if not npy_path.exists():
    process_tif_to_volume(
        tif_path=str(tif_path),
        output_path=str(npy_path),
        scale_factor=scale,
        normalize=True,
    )
else:
    print("NPY already exists:", npy_path)

## 2) Load & visualize NPY slices (use `draw_nifti_slices_with_quiver`)

In [ ]:
raw = np.load(npy_path)
print("Loaded:", npy_path)
print("raw shape:", raw.shape, "dtype:", raw.dtype, "range:", (float(raw.min()), float(raw.max())))

# draw_nifti_slices_with_quiver expects (D,H,W). If your npy is 4D, we take the first channel.
if raw.ndim == 4:
    # assume (D,H,W,C) or (C,D,H,W); this notebook expects microscopy gray volumes anyway
    if raw.shape[-1] <= 4:
        vol3d = raw[..., 0]
    else:
        vol3d = raw[0, ...]
elif raw.ndim == 3:
    vol3d = raw
else:
    raise ValueError(f"Unexpected npy shape: {raw.shape}")

_ = draw_nifti_slices_with_quiver(
    vol3d,
    slice_along_axis="x", # WARNING: animation tif use x slice to see the rotated animation.
    stride=2,
    title="Volume slices",
    cmap="gray",
)

## 3) Create a masked pretext sample & visualize input/target/mask

In [ ]:
sample = create_masked_sample(
    vol3d,
    mask_type=mask_type,
    mask_ratio=mask_ratio,
    block_size_range=(8, 32),
    patch_size=16,
 )

inp = sample["input"]  # (C,D,H,W)
tgt = sample["target"] # (C,D,H,W)
msk = sample["mask"]   # (1,D,H,W)

print("input:", tuple(inp.shape), "target:", tuple(tgt.shape), "mask:", tuple(msk.shape))
print("mask unique:", torch.unique(msk))

# Visualize: use channel 0
inp3d = inp[0]
tgt3d = tgt[0]
msk3d = msk[0]

draw_nifti_slices_with_quiver(tgt3d, mask=None, slice_along_axis="z", stride=2, title="target", cmap="gray")
draw_nifti_slices_with_quiver(inp3d, mask=None, slice_along_axis="z", stride=2, title="input (masked)", cmap="gray")
draw_nifti_slices_with_quiver(msk3d, mask=None, slice_along_axis="z", stride=2, title="mask (1=masked)", cmap="viridis")

## 4) Load default config and build a tiny `FishModule` (no training)

In [ ]:
cfg = load_config("config/default.yaml")

# Keep this notebook minimal: we only need model + training kwargs for FishModule init.
model_kwargs = dict(cfg["model"])
train_kwargs = dict(cfg["training"])

# Override to a small model so it runs fast in-notebook
model_kwargs["features"] = [16, 32]
model_kwargs["in_channels"] = 1
model_kwargs["out_channels"] = 1
train_kwargs["learning_rate"] = 1e-4
train_kwargs["weight_decay"] = 1e-5

module = FishModule(**{**model_kwargs, **train_kwargs})
module.eval()
print("Tiny model params:", module.model.get_num_params())

## 5) Infer one sample and visualize reconstruction vs target (focus on masked region)

In [ ]:
# Prepare a batch (B,C,D,H,W) from the single sample
x = inp.unsqueeze(0)  # (1,C,D,H,W)
y = tgt.unsqueeze(0)
m = msk.unsqueeze(0)

with torch.no_grad():
    pred = module(x)  # (1,C,D,H,W)

pred3d = pred[0, 0].detach().cpu()
tgt3d = y[0, 0].detach().cpu()
msk3d = m[0, 0].detach().cpu()

# Visualize: pred, target, and (pred-target) masked area
draw_nifti_slices_with_gt(pred3d.unsqueeze(-1), tgt3d.unsqueeze(-1))